# Best Image Selection Protocol Development

このノートブックは、Best画像選定アルゴリズムの開発過程を記録したものです。

## 目次

1. データ読み込みとHuman選定の取得
2. 初期アルゴリズム（ランクベース）の評価
3. Random Forestによる最適化試行
4. 問題動画の分析
5. Disc Edge Coverage指標の開発
6. 閾値最適化
7. 最終アルゴリズムの評価

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

# パス設定
BASE_DIR = Path(r'C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation')
RESULTS_DIR = BASE_DIR / 'validation_results'

print(f"BASE_DIR: {BASE_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

BASE_DIR: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation
RESULTS_DIR: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation\validation_results


## 1. データ読み込みとHuman選定の取得

In [2]:
# Human選定データの読み込み（Excelから）
xlsx_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('.xlsx')]
print(f"Found Excel files: {xlsx_files}")

df_human = pd.read_excel(RESULTS_DIR / xlsx_files[0], sheet_name=0)
print(f"\nHuman selection data shape: {df_human.shape}")
print(df_human.head())

Found Excel files: ['ベストショット一致率20260104.xlsx']

Human selection data shape: (220, 5)
   動画番号  rank  画像番号      YF      HK
0  1227     1  1095  1090.0  1090.0
1  1227     2  1100  1095.0  1095.0
2  1227     3   490  1160.0  1160.0
3  1227     4  1090  1170.0  1170.0
4  1227     5  1110  1150.0  1155.0


In [3]:
# 動画ごとのHuman選定画像を整理
human_selections = {}

for video_id in df_human['動画番号'].unique():
    video_data = df_human[df_human['動画番号'] == video_id]
    yf_images = []
    hk_images = []
    
    for _, row in video_data.iterrows():
        if pd.notna(row['YF']):
            yf_images.append(f'IMG_{video_id}_{int(row["YF"]):04d}.jpg')
        if pd.notna(row['HK']):
            hk_images.append(f'IMG_{video_id}_{int(row["HK"]):04d}.jpg')
    
    human_selections[video_id] = {
        'YF': yf_images[:5],
        'HK': hk_images[:5],
        'all': list(set(yf_images[:5] + hk_images[:5]))
    }

print(f"動画数: {len(human_selections)}")
for vid, sel in list(human_selections.items())[:3]:
    print(f"  Video {vid}: YF={len(sel['YF'])}, HK={len(sel['HK'])}, all={len(sel['all'])}")

動画数: 22
  Video 1227: YF=5, HK=5, all=6
  Video 1363: YF=5, HK=5, all=6
  Video 1376: YF=5, HK=5, all=6


In [4]:
# 全動画のvalidation resultsを読み込み
def load_all_validation_data():
    """全CSVを読み込んでマージ"""
    all_data = []
    
    for csv_file in glob.glob(str(RESULTS_DIR / "validation_results_*.csv")):
        fname = os.path.basename(csv_file)
        # disc, simple, simple+ などは除外
        if 'disc' in fname or 'simple' in fname:
            continue
        
        df = pd.read_csv(csv_file)
        if 'image_id' in df.columns:
            all_data.append(df)
    
    return pd.concat(all_data, ignore_index=True)

df_all = load_all_validation_data()
print(f"総データ数: {len(df_all)}")
print(f"動画数: {df_all['image_id'].nunique()}")
print(f"\nカラム: {list(df_all.columns)}")

総データ数: 6568
動画数: 21

カラム: ['image_id', 'image_name', 'image_path', 'lens_detected', 'lens_area', 'retina_area', 'retina_ratio', 'disc_detected', 'macula_detected', 'mbss_L_multi', 'mbss_HF_ratio', 'mbss_Spec_centroid', 'mbss_Grad_p90', 'mbss_score', 'S_mean', 'disc_core_L_multi', 'disc_core_score', 'disc_ring_L_multi', 'disc_ring_score', 'disc_center_dist_ratio', 'disc_pos_ok']


In [5]:
# Human選定ラベルを追加
def add_human_label(row):
    """Human選定画像かどうかのラベルを付与"""
    video_id = int(row['image_id'])
    image_name = row['image_name']
    
    if video_id in human_selections:
        if image_name in human_selections[video_id]['all']:
            return 1
    return 0

df_all['human_selected'] = df_all.apply(add_human_label, axis=1)

print(f"Human選定画像数: {df_all['human_selected'].sum()}")
print(f"Human選定率: {df_all['human_selected'].mean()*100:.2f}%")

Human選定画像数: 138
Human選定率: 2.10%


## 2. 初期アルゴリズム（ランクベース）の評価

In [6]:
def get_ai_top5_rank_based(df, top_k=5):
    """
    初期アルゴリズム: ランクベースのスコアリング
    
    rank_sum = 1.5 × retina_area_rank + 1.0 × mbss_rank + 0.5 × disc_ring_rank + 0.5 × s_mean_rank
    """
    # 有効データのみ
    valid = df[(df['lens_detected'] == True) & (df['retina_ratio'] > 0)].copy()
    
    if len(valid) == 0:
        return []
    
    # 各指標のランク（小さいほど良い）
    valid['retina_area_rank'] = valid['retina_area'].rank(ascending=False, method='min', na_option='bottom')
    valid['mbss_rank'] = valid['mbss_score'].rank(ascending=False, method='min', na_option='bottom')
    valid['disc_ring_rank'] = valid['disc_ring_score'].rank(ascending=False, method='min', na_option='bottom')
    valid['s_mean_rank'] = valid['S_mean'].rank(ascending=False, method='min', na_option='bottom')
    
    # 総合ランクスコア
    valid['rank_sum'] = (
        1.5 * valid['retina_area_rank'] +
        1.0 * valid['mbss_rank'] +
        0.5 * valid['disc_ring_rank'] +
        0.5 * valid['s_mean_rank']
    )
    
    # ソートして上位を返す
    valid = valid.sort_values(by=['rank_sum', 'retina_area'], ascending=[True, False])
    
    return valid.head(top_k)['image_name'].tolist()

In [ ]:
# 初期アルゴリズムの評価
def evaluate_algorithm(get_ai_top_func, df_all, human_selections, top_k=5):
    """
    アルゴリズムの評価
    
    Returns:
        image_matches: 画像一致数
        video_matches: 動画一致数（1枚以上一致）
    """
    results = []
    
    for video_id in human_selections.keys():
        df_video = df_all[df_all['image_id'] == str(video_id)]
        
        if len(df_video) == 0:
            continue
        
        ai_top = get_ai_top_func(df_video, top_k)
        human_all = set(human_selections[video_id]['all'])
        
        matches = len(set(ai_top) & human_all)
        results.append({
            'video_id': video_id,
            'ai_top': ai_top,
            'human_count': len(human_all),
            'matches': matches,
            'at_least_one': matches > 0
        })
    
    df_results = pd.DataFrame(results)
    
    total_matches = df_results['matches'].sum()
    total_possible = len(df_results) * top_k
    video_matches = df_results['at_least_one'].sum()
    
    return {
        'image_matches': total_matches,
        'image_total': total_possible,
        'image_rate': total_matches / total_possible * 100,
        'video_matches': video_matches,
        'video_total': len(df_results),
        'video_rate': video_matches / len(df_results) * 100,
        'details': df_results
    }

# 評価実行
result_initial = evaluate_algorithm(get_ai_top5_rank_based, df_all, human_selections)

print("=== 初期アルゴリズム（ランクベース）の結果 ===")
print(f"画像一致率: {result_initial['image_matches']}/{result_initial['image_total']} ({result_initial['image_rate']:.1f}%)")
print(f"動画一致率: {result_initial['video_matches']}/{result_initial['video_total']} ({result_initial['video_rate']:.1f}%)")

## 3. Random Forest による最適化試行

In [ ]:
# 特徴量の準備
feature_cols = ['retina_ratio', 'mbss_score', 'mbss_Grad_p90', 'disc_ring_score', 'S_mean']

# 有効データのみ
df_valid = df_all[(df_all['lens_detected'] == True) & (df_all['retina_ratio'] > 0)].copy()

# 欠損値を埋める
for col in feature_cols:
    if col in df_valid.columns:
        df_valid[col] = df_valid[col].fillna(df_valid[col].median())

print(f"有効データ数: {len(df_valid)}")
print(f"Human選定数: {df_valid['human_selected'].sum()}")

In [ ]:
# Random Forestで学習
X = df_valid[feature_cols].values
y = df_valid['human_selected'].values

# モデル学習
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X, y)

# Feature Importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
print(importance)

# 可視化
plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Random Forestのスコアを使った選定
def get_ai_top5_rf(df, top_k=5):
    """Random Forestの確率スコアでTop5を選定"""
    valid = df[(df['lens_detected'] == True) & (df['retina_ratio'] > 0)].copy()
    
    if len(valid) == 0:
        return []
    
    # 特徴量準備
    X_valid = valid[feature_cols].fillna(0).values
    
    # 予測確率
    valid['rf_prob'] = rf.predict_proba(X_valid)[:, 1]
    
    # ソート
    valid = valid.sort_values(by='rf_prob', ascending=False)
    
    return valid.head(top_k)['image_name'].tolist()

# 評価
result_rf = evaluate_algorithm(get_ai_top5_rf, df_all, human_selections)

print("=== Random Forest の結果 ===")
print(f"画像一致率: {result_rf['image_matches']}/{result_rf['image_total']} ({result_rf['image_rate']:.1f}%)")
print(f"動画一致率: {result_rf['video_matches']}/{result_rf['video_total']} ({result_rf['video_rate']:.1f}%)")

### 結論

Random Forestによる最適化は、大幅な精度向上には至りませんでした。
既存の指標だけではHumanの選定基準を十分に捉えられていないことが示唆されます。

## 4. 問題動画の分析

In [ ]:
# 一致率0%の動画を特定
df_details = result_initial['details']
zero_match_videos = df_details[df_details['matches'] == 0]['video_id'].tolist()

print("=== 一致率0%の動画 ===")
for vid in zero_match_videos:
    row = df_details[df_details['video_id'] == vid].iloc[0]
    print(f"Video {vid}: AI Top5 = {row['ai_top']}")
    print(f"           Human = {human_selections[vid]['all']}")
    print()

In [ ]:
# Video 1632 の詳細分析
vid = 1632
df_1632 = df_all[df_all['image_id'] == str(vid)].copy()

# AI Top5
ai_top5 = get_ai_top5_rank_based(df_1632)
ai_df = df_1632[df_1632['image_name'].isin(ai_top5)]

# Human選定
human_images = human_selections[vid]['all']
human_df = df_1632[df_1632['image_name'].isin(human_images)]

print(f"=== Video {vid} の詳細分析 ===")
print(f"\nAI Top5:")
print(ai_df[['image_name', 'retina_ratio', 'mbss_score', 'disc_center_dist_ratio']].to_string())

print(f"\nHuman選定:")
print(human_df[['image_name', 'retina_ratio', 'mbss_score', 'disc_center_dist_ratio']].to_string())

In [ ]:
# フレーム番号の分布を確認
def extract_frame_number(image_name):
    return int(image_name.split('_')[-1].replace('.jpg', ''))

ai_frames = sorted([extract_frame_number(img) for img in ai_top5])
human_frames = sorted([extract_frame_number(img) for img in human_images])

print(f"\nVideo {vid}:")
print(f"  AI Top5 frames:    {ai_frames}")
print(f"  Human frames:      {human_frames}")
print(f"  AI range:          {min(ai_frames)} - {max(ai_frames)}")
print(f"  Human range:       {min(human_frames)} - {max(human_frames)}")

### Video 1632 の分析結果

- **AI選定画像**: 動画後半のフレーム（retina_ratio高い）
- **Human選定画像**: 動画前半のフレーム（retina_ratio低め）
- **違いの原因**: Discの位置・辺縁の被覆状態

## 5. Disc Edge Coverage指標の開発

In [ ]:
# Disc指標を含むCSVを読み込み
def load_with_disc_metrics(video_id):
    """Original CSVとDisc CSVをマージ"""
    orig_path = RESULTS_DIR / f'validation_results_{video_id}.csv'
    disc_path = RESULTS_DIR / f'validation_results_disc_{video_id}.csv'
    
    df_orig = pd.read_csv(orig_path)
    
    if disc_path.exists():
        df_disc = pd.read_csv(disc_path)
        disc_cols = ['disc_edge_covered', 'disc_edge_coverage_ratio', 'disc_area', 'disc_area_ratio']
        existing_cols = [c for c in disc_cols if c in df_disc.columns]
        if existing_cols:
            df = df_orig.merge(df_disc[['image_name'] + existing_cols], on='image_name', how='left')
            return df
    
    return df_orig

# 全動画のデータを読み込み
df_all_disc = []
for vid in human_selections.keys():
    df_vid = load_with_disc_metrics(vid)
    df_all_disc.append(df_vid)

df_all_disc = pd.concat(df_all_disc, ignore_index=True)
print(f"総データ数: {len(df_all_disc)}")

In [ ]:
# Human選定画像のdisc_edge_coverage_ratioを確認
df_all_disc['human_selected'] = df_all_disc.apply(add_human_label, axis=1)

human_disc_cov = df_all_disc[df_all_disc['human_selected'] == 1]['disc_edge_coverage_ratio'].dropna()
all_disc_cov = df_all_disc['disc_edge_coverage_ratio'].dropna()

print("=== disc_edge_coverage_ratio の分布 ===")
print(f"\nHuman選定画像:")
print(f"  平均: {human_disc_cov.mean():.3f}")
print(f"  中央値: {human_disc_cov.median():.3f}")
print(f"  最小: {human_disc_cov.min():.3f}")
print(f"  最大: {human_disc_cov.max():.3f}")

print(f"\n全画像:")
print(f"  平均: {all_disc_cov.mean():.3f}")
print(f"  中央値: {all_disc_cov.median():.3f}")

In [ ]:
# ヒストグラムで比較
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(all_disc_cov, bins=50, alpha=0.5, label='All images', density=True)
ax.hist(human_disc_cov, bins=50, alpha=0.7, label='Human selected', density=True)

ax.axvline(x=0.80, color='r', linestyle='--', label='Threshold 0.80')
ax.set_xlabel('disc_edge_coverage_ratio')
ax.set_ylabel('Density')
ax.set_title('Distribution of disc_edge_coverage_ratio')
ax.legend()
plt.tight_layout()
plt.show()

## 6. 閾値最適化

In [ ]:
def get_ai_top5_with_cutoff(df, cutoff=0.80, top_k=5):
    """
    足切り + スコアリング方式
    
    1. disc_edge_coverage_ratio >= cutoff で足切り
    2. score = 0.4*retina + 0.4*Grad_p90 + 0.2*mbss でスコアリング
    3. 足りなければ retina_ratio のみでソートして補完
    """
    # 有効データ
    valid = df[
        (df['lens_detected'] == True) & 
        (df['retina_ratio'] > 0) & 
        (df['disc_detected'] == True)
    ].copy()
    
    if len(valid) == 0:
        return []
    
    # 正規化関数
    def minmax_norm(series):
        min_val = series.min()
        max_val = series.max()
        if max_val - min_val < 1e-8:
            return pd.Series([0.5] * len(series), index=series.index)
        return (series - min_val) / (max_val - min_val)
    
    # Stage 1: 足切り通過
    if 'disc_edge_coverage_ratio' in valid.columns:
        stage1 = valid[valid['disc_edge_coverage_ratio'] >= cutoff].copy()
    else:
        stage1 = valid.copy()
    
    if len(stage1) > 0:
        stage1['retina_norm'] = minmax_norm(stage1['retina_ratio'].fillna(0))
        stage1['grad_norm'] = minmax_norm(stage1['mbss_Grad_p90'].fillna(0))
        stage1['mbss_norm'] = minmax_norm(stage1['mbss_score'].fillna(0))
        
        stage1['score'] = 0.4 * stage1['retina_norm'] + 0.4 * stage1['grad_norm'] + 0.2 * stage1['mbss_norm']
        stage1 = stage1.sort_values(by='score', ascending=False)
        selected = stage1.head(top_k)
    else:
        selected = pd.DataFrame()
    
    # Stage 2: 補完
    n_remaining = top_k - len(selected)
    if n_remaining > 0:
        remaining = valid[~valid.index.isin(selected.index)].copy()
        if len(remaining) > 0:
            remaining = remaining.sort_values(by='retina_ratio', ascending=False)
            selected = pd.concat([selected, remaining.head(n_remaining)])
    
    return selected['image_name'].tolist()

In [ ]:
# 閾値を変えて評価
thresholds = [0.75, 0.80, 0.85, 0.90, 0.95, 0.98]
results_by_threshold = []

for th in thresholds:
    def get_ai_func(df, top_k=5):
        return get_ai_top5_with_cutoff(df, cutoff=th, top_k=top_k)
    
    result = evaluate_algorithm(get_ai_func, df_all_disc, human_selections)
    results_by_threshold.append({
        'threshold': th,
        'image_matches': result['image_matches'],
        'image_rate': result['image_rate'],
        'video_matches': result['video_matches'],
        'video_rate': result['video_rate']
    })

df_threshold = pd.DataFrame(results_by_threshold)
print("=== 閾値別の結果 ===")
print(df_threshold.to_string(index=False))

In [ ]:
# 可視化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(df_threshold['threshold'], df_threshold['image_rate'], 'o-', label='Image Match Rate')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Image Match Rate (%)')
ax1.set_title('Image Match Rate vs Threshold')
ax1.axvline(x=0.80, color='r', linestyle='--', alpha=0.5)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df_threshold['threshold'], df_threshold['video_rate'], 's-', label='Video Match Rate', color='orange')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Video Match Rate (%)')
ax2.set_title('Video Match Rate vs Threshold')
ax2.axvline(x=0.80, color='r', linestyle='--', alpha=0.5)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n最適閾値: 0.80 (画像一致率が最大)")

## 7. 最終アルゴリズムの評価

In [ ]:
# 最終アルゴリズム（閾値0.80）で評価
def get_ai_top5_final(df, top_k=5):
    return get_ai_top5_with_cutoff(df, cutoff=0.80, top_k=top_k)

result_final = evaluate_algorithm(get_ai_top5_final, df_all_disc, human_selections)

print("=" * 60)
print("最終アルゴリズムの結果")
print("=" * 60)
print(f"\n画像一致率: {result_final['image_matches']}/{result_final['image_total']} ({result_final['image_rate']:.1f}%)")
print(f"動画一致率: {result_final['video_matches']}/{result_final['video_total']} ({result_final['video_rate']:.1f}%)")

In [ ]:
# 初期アルゴリズムとの比較
print("=" * 60)
print("精度比較")
print("=" * 60)
print(f"\n{'指標':<20} {'初期':<20} {'最終':<20} {'改善':<10}")
print("-" * 70)
print(f"{'画像一致率':<20} {result_initial['image_rate']:.1f}%{' '*15} {result_final['image_rate']:.1f}%{' '*15} +{result_final['image_rate'] - result_initial['image_rate']:.1f}%")
print(f"{'動画一致率':<20} {result_initial['video_rate']:.1f}%{' '*15} {result_final['video_rate']:.1f}%{' '*15} +{result_final['video_rate'] - result_initial['video_rate']:.1f}%")

In [ ]:
# 動画別の詳細結果
print("\n=== 動画別詳細 ===")
df_detail = result_final['details'].sort_values('matches')

for _, row in df_detail.iterrows():
    status = "✓" if row['at_least_one'] else "✗"
    print(f"{status} Video {row['video_id']}: {row['matches']}/5 一致")

In [ ]:
# 一致率0%の動画の分析
zero_match = df_detail[df_detail['matches'] == 0]

print("\n=== 一致率0%の動画の原因分析 ===")
for _, row in zero_match.iterrows():
    vid = row['video_id']
    print(f"\nVideo {vid}:")
    print(f"  AI Top5: {row['ai_top']}")
    print(f"  Human:   {human_selections[vid]['all']}")
    
    # Human選定フレームがデータに存在するか確認
    df_vid = df_all_disc[df_all_disc['image_id'] == str(vid)]
    existing = [img for img in human_selections[vid]['all'] if img in df_vid['image_name'].values]
    print(f"  データに存在するHuman選定: {len(existing)}/{len(human_selections[vid]['all'])}")

## まとめ

### 開発過程

1. **初期アルゴリズム（ランクベース）**: 画像一致率 ~43%
2. **Random Forest最適化**: 大幅な改善なし
3. **問題動画の分析**: Disc辺縁の被覆状態が重要と判明
4. **Disc Edge Coverage指標の開発**: 新指標を導入
5. **閾値最適化**: 0.80が最適
6. **最終アルゴリズム**: 画像一致率 54.5%、動画一致率 86.4%

### 最終アルゴリズム

1. **足切り**: `disc_edge_coverage_ratio >= 0.80`
2. **スコアリング**: `score = 0.4 × retina + 0.4 × Grad_p90 + 0.2 × mbss`
3. **補完**: 足切りで不足する場合は `retina_ratio` のみでソート

### 残存する問題

- **Video 1632**: シーン選好の違い（時間的な違い）
- **Video 1732**: データサンプリングの問題